# 7. Визуализация: от EVT-сигнала к начальному источнику

**Цель:** на многоканальном ряде показать одной проверяемой фигурой (1) индикатор и POT/GPD-порог, (2) момент устойчивого EVT-превышения, (3) отклики каналов рядом с ним и (4) ранжирование предполагаемого начального источника. Красный столбец — оценка модели, а не доказанная причинность.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig
from evt_demo.visualization import plot_evt_source_result

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/kuramoto_synchronized_series.csv").exists())
DATA = ROOT / "data/generated/kuramoto_synchronized_series.csv"
SEED = 42
df = pd.read_csv(DATA)
channel_names = [column for column in df if column.startswith("channel_")]
x = df[channel_names].to_numpy()
assert np.isfinite(x).all() and x.ndim == 2

In [ ]:
BASELINE_SIZE = 300
pipeline = GraphEVTPipeline(
    EVTConfig(baseline_size=BASELINE_SIZE, tail_quantile=0.90, alarm_probability=0.99, persistence=2),
    graph=GraphConfig(random_state=SEED),
    input_config=InputConfig(missing="error"),
)
result = pipeline.run_detailed(x)
assert result.detection.detected and result.ranking is not None
{"detected_at": result.detection.time_index, "estimated_source": channel_names[result.ranking.source], "probability": float(result.ranking.probabilities[0])}

In [ ]:
figure, axes = plot_evt_source_result(
    x, result, timestamps=df["timestamp"].to_numpy(), channel_names=channel_names,
    baseline_size=BASELINE_SIZE, context=80, output_path=ROOT / "figures/evt_source_result.png",
)

На верхней панели пересечение синего индикатора с красным порогом объясняет детекцию. Тепловая карта показывает нормированные реакции, использованные после тревоги. Нижняя панель показывает **всё распределение** рейтинга: первый узел — оценённый исходный источник. В метаданных генератора номинальный frequency-lead задан отдельно; несовпадение с ranking следует обсуждать как ошибку/ограничение локализации, а не скрывать.